In [6]:
import boto3, botocore

print(boto3.client('sts').get_caller_identity()['Arn'])
print(boto3.session.Session().region_name)
print(boto3.__version__, botocore.__version__)

arn:aws:iam::317804964135:user/ereza-lab
us-west-1
1.43.127 1.43.127


In [25]:
control = boto3.client('bedrock')
models = control.list_foundation_models()['modelSummaries']
print(len(models), 'models listed in this Region')

for m in models[:10]:
    print(m['modelId'], m['providerName'])

runtime = boto3.client('bedrock-runtime', region_name='us-west-1')
resp = runtime.converse(
    modelId='us.openai.gpt-6-astra',
    messages=[{'role': 'user',
               'content': [{'text': "yo what's up gang"}]}],
    inferenceConfig={'maxTokens': 200},
)

print(resp)              # the whole thing, once
print(resp['usage'])
print(resp['output']['message']['content'][0]['text'])

44 models listed in this Region
anthropic.claude-opus-5 Anthropic
openai.gpt-6-astra OpenAI
openai.gpt-5.6-terra OpenAI
anthropic.claude-opus-4-8 Anthropic
anthropic.claude-opus-4-7 Anthropic
anthropic.claude-haiku-4-5-20251001-v1:0 Anthropic
anthropic.claude-opus-5-5 Anthropic
openai.gpt-6-sol OpenAI
moonshotai.kimi-k3.1 Moonshot AI
writer.palmyra-x4-v1:0 Writer


AccessDeniedException: An error occurred (AccessDeniedException) when calling the Converse operation: openai.gpt-6-astra is not available for this account. You can explore other available models on Amazon Bedrock. For additional access options, contact AWS Sales at https://aws.amazon.com/contact-us/sales-support/

In [ ]:
ticket = "I would like a refund. Order #9931"

prose = (
    f"Ticket: '{ticket}'\n"
    "Classify this support message into a department. "
    "Write a detailed paragraph explaining your routing decision and reasoning."
)

structured = (
    f"Ticket: '{ticket}'\n"
    "Classify this support message into a department. "
    "Return ONLY a JSON object with two fields: 'department' and 'confidence'."
)

for name, prompt in [('prose', prose), ('structured', structured)]:
    r = runtime.converse(
        modelId='us.amazon.nova-2-lite-v1:0',
        messages=[{'role': 'user', 'content': [{'text': prompt}]}],
        inferenceConfig={'maxTokens': 5},   
    )
    print(name, r['usage'])

prose {'inputTokens': 79, 'outputTokens': 5, 'totalTokens': 84}
structured {'inputTokens': 84, 'outputTokens': 5, 'totalTokens': 89}


In [23]:
in_tokens  = 84      # your measured number
out_tokens = 24      # your measured number
in_rate    = 0.00039      # dollars per 1,000 input tokens
out_rate   = 0.00321      # dollars per 1,000 output tokens

per_call = (in_tokens / 1000 * in_rate) + (out_tokens / 1000 * out_rate)
print('per call', per_call)

for volume in [1000, 50000, 2000000]:
    print(volume, 'calls a month costs', volume * per_call)

per call 0.00010980000000000001
1000 calls a month costs 0.10980000000000001
50000 calls a month costs 5.49
2000000 calls a month costs 219.60000000000002


In [27]:
# BREAK IT

control = boto3.client('bedrock')
runtime = boto3.client('bedrock-runtime')

# Attempt 1: Calling converse() on the control plane client
control.converse(
    modelId='amazon.nova-lite-v1:0',
    messages=[{'role': 'user', 'content': [{'text': 'hi'}]}],
    inferenceConfig={'maxTokens': 100},
)

AttributeError: 'Bedrock' object has no attribute 'converse'